# SHAP Explainability for the Saved Random Forest

This notebook explains the already trained Random Forest using SHAP 0.52.0. It uses the exact saved training feature list and the saved test-student split. It does not retrain a model or modify either dataset.

SHAP indicates how features contributed to a model prediction; it does not establish that a feature caused the outcome.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import shap

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from explainability import load_explainability_inputs, _multiclass_values, run_explainability

data, model, feature_columns, test_data, X_test, y_test = load_explainability_inputs()
print('SHAP version:', shap.__version__)
print('Model:', type(model).__name__)
print('Features explained:', len(feature_columns))
print('Test rows:', len(X_test))

## 1. Feature and Leakage Checks

In [ ]:
forbidden_columns = {'student_id', 'academic_difficulty', 'future_mean_topic_score', 'future_score_change', 'week', 'subject', 'topic'}
print('student_id excluded:', 'student_id' not in feature_columns)
print('academic_difficulty excluded:', 'academic_difficulty' not in feature_columns)
print('future/raw repeated-record columns excluded:', not forbidden_columns.intersection(feature_columns))
print('Exact saved training feature matrix used:', list(X_test.columns) == feature_columns)
print('Input features are weeks 1-6 engineered features only.')

## 2. Global and Local SHAP Explanations

In [ ]:
print('Running SHAP TreeExplainer for the saved Random Forest...')
run_explainability()

## Interpretation

The global ranking summarizes mean absolute SHAP values across the multiclass outputs. The class-specific CSV ranks contributions separately for Low, Moderate, and High. The local figure uses a saved test-set student and shows positive and negative contributions for the class predicted by the Random Forest.

These explanations describe the model's behavior. They are not causal evidence and should not be interpreted as proof that any learning behavior caused academic difficulty.